# Oracle Basics — Amazon Braket

A **phase oracle** flips the sign of the target computational basis
state while leaving all others unchanged.  Here we mark $|11\rangle$
with a CZ gate and verify the phase flip by inspecting amplitudes.

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Oracle on $|00\rangle$

CZ applied to $|00\rangle$ does nothing — $|11\rangle$ has zero
amplitude.

In [ ]:
circuit = Circuit()
circuit.cz(0, 1)

result = device.run(circuit, shots=0).result()
amps = result.result_types[0].value
probs = result.result_types[1].value

print(f"amplitudes: {amps}")
print(f"probabilities: {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
print("No phase flip — |11> is not present in the state.")

## Oracle on equal superposition

$H(0)\,H(1)$ creates $\frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$.
The CZ oracle flips only the $|11\rangle$ component.

In [ ]:
circuit = Circuit()
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)

result = device.run(circuit, shots=0).result()
amps = result.result_types[0].value
probs = result.result_types[1].value

print(f"amplitudes: {amps}")
print(f"probabilities: {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
print("Only |11> has its sign flipped — the others are unchanged.")

## Marking different states

We can mark any target state by wrapping the CZ with X gates on the
qubits that should be $|0\rangle$.

In [ ]:
def mark_state(target: int, circuit: Circuit) -> None:
    """Mark the given 2-bit state (MSB=q1, LSB=q0) with a phase flip."""
    bits = format(target, '02b')
    if bits[1] == '0':
        circuit.x(0)
    if bits[0] == '0':
        circuit.x(1)
    circuit.cz(0, 1)
    if bits[1] == '0':
        circuit.x(0)
    if bits[0] == '0':
        circuit.x(1)

for target in [2, 1]:
    label = f"|{format(target, '02b')}>"
    print(f"=== Oracle marking {label} ===")
    circuit = Circuit()
    circuit.h(0)
    circuit.h(1)
    mark_state(target, circuit)
    result = device.run(circuit, shots=0).result()
    amps = result.result_types[0].value
    print(f"amplitudes: {amps}")
    probs = result.result_types[1].value
    print(f"probabilities: {json.dumps({k: round(v, 6) for k, v in probs.items()})}")
    print()